# Run Kubeflow Pipelines from JupyterLab with Elyra

Elyra lets you build a visual pipeline in JupyterLab by connecting notebooks on a canvas, then submitting the pipeline to Kubeflow Pipelines (KFP). This notebook walks through a two-notebook hello-world pipeline and shows how to verify the run in the KFP UI.

This workflow uses KFP v2. Each notebook node is executed as a containerized pipeline step, and KFP stores run artifacts in S3-compatible object storage.

## Configure Elyra first

Before creating or running a pipeline, run the [Elyra Runtime and Runtime Images configuration notebook](./assets/elyra-runtime-configuration.ipynb) in the same JupyterLab Workbench. It creates the KFP Runtime and selectable Runtime Image metadata in the Workbench user's Jupyter data directory.

> **Administrator prerequisite for air-gapped clusters:** Complete the administrator setup in the next section before users submit a pipeline.

For an air-gapped cluster, complete the image mirroring steps in that notebook first. The runtime configuration notebook uses cluster-internal KFP and object-storage endpoints and internal registry image references, so this pipeline notebook does not repeat those commands. Refresh JupyterLab after the configuration notebook completes.

## Administrator setup for an air-gapped cluster

Elyra's KFP processor generates each generic node with shell commands that download runtime helper files with `curl`. By default, the URLs point to the Elyra GitHub repository, which is not usable in a disconnected cluster. Follow the upstream [Running Elyra in an air-gapped environment](https://elyra.readthedocs.io/en/latest/recipes/running-elyra-in-air-gapped-environment.html) guidance and relocate the files to a service that KFP pods can reach.

### 1. Prepare version-matched files

Use the same Elyra version as the JupyterLab Workbench image. For the Alauda images documented here, the files are from Elyra `v4.3.1`. The required KFP files are:

- `elyra/kfp/bootstrapper.py` - executes each notebook or script and transfers inputs, outputs, metrics, and metadata.
- `etc/generic/requirements-elyra.txt` - Elyra execution dependencies. This is the file name Elyra expects; it is not the Workbench image's general `requirements.txt`.
- `etc/kfp/pip.conf` - required when the KFP runtime uses the CRI-O user-volume layout. It configures pip to install into `/opt/app-root/src/jupyter-work-dir/python3`.

On a connected staging host, download the files and transfer the reviewed copies to the disconnected environment:

```bash
ELYRA_VERSION=v4.3.1
mkdir -p elyra-airgap-assets/elyra/kfp elyra-airgap-assets/etc/generic elyra-airgap-assets/etc/kfp
curl -fL https://raw.githubusercontent.com/opendatahub-io/elyra/${ELYRA_VERSION}/elyra/kfp/bootstrapper.py \
  -o elyra-airgap-assets/elyra/kfp/bootstrapper.py
curl -fL https://raw.githubusercontent.com/opendatahub-io/elyra/${ELYRA_VERSION}/etc/generic/requirements-elyra.txt \
  -o elyra-airgap-assets/etc/generic/requirements-elyra.txt
curl -fL https://raw.githubusercontent.com/opendatahub-io/elyra/${ELYRA_VERSION}/etc/kfp/pip.conf \
  -o elyra-airgap-assets/etc/kfp/pip.conf
```

Do not download these files during a pipeline run. Review and checksum them during the connected staging step, then transfer them through your approved offline media process.

### 2. Serve the files over anonymous HTTP

Place the three files at stable paths in an S3-compatible bucket or another internal HTTP server. A common arrangement is a public-read object in a bucket exposed through the cluster's object-storage HTTP endpoint. Public-read means anonymous `GET`; it does not mean that the cluster or bucket must be exposed to the internet. Keep the bucket limited to these non-secret helper files and use a versioned prefix.

For an S3-compatible CLI, the upload is equivalent to the following. Adapt the endpoint, credentials, and public-read policy to your object-storage product; some products use a bucket policy instead of `--acl public-read`:

```bash
S3_ENDPOINT=https://s3.<cluster-domain>
S3_BUCKET=elyra-public
aws --endpoint-url $S3_ENDPOINT s3 cp elyra-airgap-assets/elyra/kfp/bootstrapper.py \
  s3://$S3_BUCKET/elyra/v4.3.1/elyra/kfp/bootstrapper.py --acl public-read
aws --endpoint-url $S3_ENDPOINT s3 cp elyra-airgap-assets/etc/generic/requirements-elyra.txt \
  s3://$S3_BUCKET/elyra/v4.3.1/etc/generic/requirements-elyra.txt --acl public-read
aws --endpoint-url $S3_ENDPOINT s3 cp elyra-airgap-assets/etc/kfp/pip.conf \
  s3://$S3_BUCKET/elyra/v4.3.1/etc/kfp/pip.conf --acl public-read
```

The resulting URLs must be usable by both the Workbench pod and KFP implementation pods, for example:

```text
https://s3.<cluster-domain>/elyra-public/elyra/v4.3.1/elyra/kfp/bootstrapper.py
https://s3.<cluster-domain>/elyra-public/elyra/v4.3.1/etc/generic/requirements-elyra.txt
https://s3.<cluster-domain>/elyra-public/elyra/v4.3.1/etc/kfp/pip.conf
```

Verify anonymous access from a Workbench and from a temporary pod scheduled in the KFP namespace. A successful response must return the file without authentication or an internet redirect:

```bash
curl --fail --location --silent --show-error -o /dev/null https://s3.<cluster-domain>/elyra-public/elyra/v4.3.1/elyra/kfp/bootstrapper.py
curl --fail --location --silent --show-error -o /dev/null https://s3.<cluster-domain>/elyra-public/elyra/v4.3.1/etc/generic/requirements-elyra.txt
```

If the object store requires authentication, use an internal unauthenticated read-only proxy for these files instead. Elyra's generated `curl` command does not send object-storage credentials.

### 3. Set the URLs in the Workbench `WorkspaceKind`

Set the URL variables on the JupyterLab Workbench `WorkspaceKind`, not only on an individual notebook. `spec.podTemplate.extraEnv` puts them in the JupyterLab process; when the user submits a pipeline, Elyra reads these values and writes them into the generated KFP command. The URLs must therefore be reachable from KFP pods as well. Preserve existing `extraEnv` entries such as `NB_PREFIX` and `NOTEBOOK_BASE_URL`.

Add these entries to the `spec.podTemplate.extraEnv` list of the JupyterLab `WorkspaceKind` used by Elyra:

```yaml
apiVersion: kubeflow.org/v1beta1
kind: WorkspaceKind
metadata:
  name: <jupyterlab-workspacekind>
spec:
  podTemplate:
    extraEnv:
      - name: ELYRA_BOOTSTRAP_SCRIPT_URL
        value: https://s3.<cluster-domain>/elyra-public/elyra/v4.3.1/elyra/kfp/bootstrapper.py
      - name: ELYRA_REQUIREMENTS_URL
        value: https://s3.<cluster-domain>/elyra-public/elyra/v4.3.1/etc/generic/requirements-elyra.txt
      - name: ELYRA_PIP_CONFIG_URL
        value: https://s3.<cluster-domain>/elyra-public/elyra/v4.3.1/etc/kfp/pip.conf
```

Back up the resource, merge the three entries into its existing `extraEnv`, and apply the change. Do not replace the complete list if it contains platform-specific variables:

```bash
kubectl get workspacekind <jupyterlab-workspacekind> -o yaml > workspacekind-before-elyra-airgap.yaml
kubectl apply -f <updated-workspacekind.yaml>
kubectl get workspacekind <jupyterlab-workspacekind> -o jsonpath='{.spec.podTemplate.extraEnv}'
```

Restart existing Workbench pods after changing the `WorkspaceKind`; new pods inherit the values automatically. Confirm the values from a JupyterLab terminal:

```bash
env | grep '^ELYRA_.*URL='
```

### 4. Prevent package installation during disconnected runs

The bootstrapper reads `ELYRA_INSTALL_PACKAGES` inside the KFP runtime container. Its default is `true`, which makes the step compare the running image with `requirements-elyra.txt` and invoke pip. In a disconnected cluster, either provide an internal package index, or preinstall the requirements into each mirrored runtime image and set `ELYRA_INSTALL_PACKAGES=false` in that image. The Alauda pipeline runtime images already include the Elyra helper files and set this variable to `false`.

A custom runtime image should contain at least `python3`, `curl`, `packaging`, the packages listed in `requirements-elyra.txt`, and (for CRI-O) the matching `pip.conf`. For example:

```dockerfile
COPY bootstrapper.py requirements-elyra.txt pip.conf /opt/app-root/bin/utils/
ENV ELYRA_INSTALL_PACKAGES="false"
```

Even when package installation is disabled, the generated command still looks for the helper files. Keep all three files under the configured file base path, or continue serving their URLs from the `WorkspaceKind`.

### Alternative: bake the helper files into the runtime image

For KFP, Elyra checks the runtime container before downloading the URLs. If every mirrored runtime image contains `/opt/app-root/bin/utils/bootstrapper.py`, `/opt/app-root/bin/utils/requirements-elyra.txt`, and `/opt/app-root/bin/utils/pip.conf`, the default `ELYRA_FILE_BASE_PATH=/opt/app-root/bin/utils` is sufficient and the URL variables are not required for those runtime images. If the files are stored elsewhere, set `ELYRA_FILE_BASE_PATH` in the runtime container to that directory. Do not set only `ELYRA_FILE_BASE_PATH` on the Workbench `WorkspaceKind`: it must be present in the KFP runtime pod, for example as an image `ENV` or a node environment variable.

## Prerequisites

- Alauda AI Workbench is installed and you can create or open a JupyterLab Workbench.
- Kubeflow Base (`kfbase`) and Kubeflow Pipelines (`kfp`) are deployed.
- Your namespace is visible in Kubeflow and bound to your user.
- KFP is configured to store artifacts in object storage.
- The Workbench image includes Elyra and the KFP SDK 2.x, for example Standard Data Science.
- The [Elyra Runtime and Runtime Images configuration notebook](./assets/elyra-runtime-configuration.ipynb) completed successfully.

The code-server Workbench image does not provide the Elyra visual pipeline editor. Use a JupyterLab image for this workflow.

## Verify the Workbench

1. Log in to Alauda AI and go to **Workbench**.
2. Open an existing JupyterLab Workbench or create one with an Elyra-enabled JupyterLab image.
3. Wait for the Workbench status to become `Running`, then click **Connect**.
4. Open a JupyterLab terminal and verify the KFP SDK version:

   ```bash
   python -c "import kfp; print(kfp.__version__)"
   ```

   The version should be `2.x`. Open the Elyra **Runtime Images** panel and confirm that at least one image created by the configuration notebook is available.

## Create the demo notebooks

Create a folder named `hello-two-nodes` in the JupyterLab file browser. Add these two notebooks to the folder.

### `01-hello.ipynb`

Add and run this cell:

```python
print("hello from the first Elyra node")
message = "first notebook completed"
print(message)
```

### `02-world.ipynb`

Add and run this cell:

```python
print("hello from the second Elyra node")
print("this notebook runs after the first notebook succeeds")
```

Run both notebooks once in JupyterLab to confirm that they execute locally without syntax errors.

## Create the Elyra pipeline

1. Open the `hello-two-nodes` folder and open the **Launcher** tab. If it is not visible, select **File** > **New Launcher**.
2. Click **Pipeline Editor** and save the empty pipeline as `hello-two-nodes.pipeline` in the folder.
3. Drag `01-hello.ipynb` and `02-world.ipynb` from the file browser onto the canvas. Arrange them from left to right.
4. Connect the output port of `01-hello.ipynb` to the input port of `02-world.ipynb`. This edge makes the second notebook start only after the first succeeds.

## Configure node properties

1. Select the `01-hello.ipynb` node and open its properties panel.
2. Confirm that the node file points to `01-hello.ipynb`.
3. Set **Runtime Image** to an image created by the runtime configuration notebook, for example `Runtime | Minimal | CPU | Python 3.12`.
4. Keep CPU, memory, GPU, environment variable, input file, and output file fields at their defaults for this hello-world example.
5. Select `02-world.ipynb` and set the same **Runtime Image**.
6. Save the pipeline again.

For this example, the connection is only an execution dependency. If the second notebook must consume a file created by the first, configure the node file dependencies and outputs so Elyra transfers the file through the KFP artifact store.

## Submit the pipeline to KFP

1. In the Elyra pipeline editor, click **Run Pipeline**.
2. Set **Runtime Platform** to **Kubeflow Pipelines**.
3. Select the KFP v2 Runtime created by the configuration notebook, for example `MLOps KFP (air-gapped)`.
4. Enter the pipeline name `hello-two-nodes`, a run name, and an existing or new experiment such as `elyra-demo`.
5. Review both node Runtime Image values. In an air-gapped cluster, these must be the internal registry references configured by the runtime notebook.
6. Click **OK** or **Submit**.

When submission succeeds, Elyra shows a dialog similar to `Job submission to Pipelines succeeded`. Elyra is an authoring and submission UI; use Kubeflow Pipelines to inspect durable run history.

## Verify the run

1. Open the Kubeflow UI and select the same namespace as the Workbench.
2. Go to **Pipelines** > **Runs** and open the latest `hello-two-nodes` run.
3. In the **Graph** tab, confirm that the two notebook nodes appear in order.
4. Check each node's logs. The output should include:

   ```text
   hello from the first Elyra node
   hello from the second Elyra node
   ```

5. Wait until the run status becomes `Succeeded`.

You can also inspect resources from a terminal:

```bash
kubectl get pod -n <your-namespace>
kubectl get workflow -n <your-namespace>
```

For KFP v2, driver and implementation pods are expected. The notebook code runs in the implementation container.

## Troubleshooting

### The namespace is not visible in Kubeflow

The namespace must be associated with a Kubeflow `Profile` and your user must be bound to it. Ask the platform administrator to check the Kubeflow namespace binding.

### Elyra does not show a KFP Runtime

Run the [Elyra Runtime and Runtime Images configuration notebook](./assets/elyra-runtime-configuration.ipynb) again in this Workbench, then refresh JupyterLab. Confirm that the KFP endpoint, namespace, object-storage endpoint, bucket, and credentials are cluster-internal and correct.

### A Runtime Image is not available

Run the configuration notebook's verification cell. In a private registry, confirm that the Runtime Image metadata uses the internal image address and pull-secret name, and that the pull Secret exists in the pipeline namespace.

### The run fails before notebook code starts

Check the namespace-side KFP objects and the failing pod logs:

```bash
kubectl get secret -n <your-namespace> mlpipeline-minio-artifact
kubectl get configmap -n <your-namespace> kfp-launcher
kubectl get configmap -n <your-namespace> metadata-grpc-configmap
```

Storage failures usually indicate an object-storage endpoint, bucket, or credential problem. Image pull failures usually indicate an unavailable internal registry or missing pull Secret. Metadata reporting failures usually indicate a `metadata-grpc-configmap` or metadata gRPC service problem.

### The Elyra success dialog was closed

Open Kubeflow Pipelines directly and check **Runs** in the same namespace. Elyra does not keep a persistent run link after the success dialog is closed.